# Notebook 02 — Exploratory Data Analysis

**Project:** TCO Optimisation Model — Sprint One  
**Author:** Soham Dharne (2026)  
**NFR compliance:** NFR-09 (100% annotated)

---

## 1. Purpose of EDA

Exploratory Data Analysis (EDA) serves three roles before modelling:

1. **Validation** — confirm the synthesised dataset has no structural errors (wrong dtypes, unexpected nulls, out-of-range values, invalid ordinal levels)
2. **Signal discovery** — identify which features have the strongest discriminative power for the classification target and predictive power for the regression target
3. **Engineering inputs** — surface collinear features, skewed distributions, or leakage risks that inform the preprocessing decisions made in Notebook 03

We use `utils.validate_dataframe` for programmatic validation and `matplotlib`/`seaborn` for visual analysis.

## 2. Environment Setup

In [ ]:
import sys
import os

NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f'Project root : {PROJECT_ROOT}')
print(f'src/ on path : {SRC_DIR}')

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from utils import validate_dataframe
from config import (
    NUMERIC_FEATURES, ORDINAL_FEATURES, BINARY_FEATURES,
    NOMINAL_FEATURES, DOMAIN_CATEGORIES,
    TARGET_CLASS, TARGET_REGR, CLASS_LABELS,
    DATASET_PATH,
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

print('Libraries loaded successfully')

## 3. Load Dataset

We load the canonical CSV produced by Notebook 01. Using `DATASET_PATH` from `config.py` ensures we always read from the correct location regardless of the working directory.

In [ ]:
df = pd.read_csv(DATASET_PATH)

print(f'Path    : {DATASET_PATH}')
print(f'Shape   : {df.shape}')
print(f'Columns : {list(df.columns)}')
print()
df.head(3)

## 4. Programmatic Validation with `validate_dataframe`

`validate_dataframe` from `utils.py` performs six checks:
1. **Missing values** — any null in the 17 input features
2. **Negative numerics** — catches data corruption in `estimated_loc`, `timeline_days`, etc.
3. **Ordinal level validity** — ensures each ordinal value is within the configured level set
4. **Binary column** — `regulatory_compliance` must be strictly 0 or 1
5. **Domain categories** — `domain_category` must be one of the 5 configured values
6. **Class label validity** — `target_team_label` must be Human / Hybrid / AI

An empty error list means the dataset passed all checks.

In [ ]:
errors = validate_dataframe(df)

if errors:
    print(f'VALIDATION FAILED — {len(errors)} issue(s) found:')
    for e in errors:
        print(f'  [ERROR] {e}')
else:
    print('VALIDATION PASSED — dataset is clean')
    print(f'  Records  : {len(df):,}')
    print(f'  Features : {len(NUMERIC_FEATURES) + len(ORDINAL_FEATURES) + len(BINARY_FEATURES) + len(NOMINAL_FEATURES)}')
    print(f'  Nulls    : {df.isnull().sum().sum()}')

## 5. Dtypes and Missing Values Summary

We confirm that numeric columns have the correct dtype (int64/float64) and string columns are `object`. Mixing dtypes would cause silent downstream errors in `OrdinalEncoder` or `StandardScaler`.

In [ ]:
info = pd.DataFrame({
    'dtype':   df.dtypes,
    'non_null': df.notnull().sum(),
    'null':    df.isnull().sum(),
    'unique':  df.nunique(),
})
info

## 6. Class Balance Analysis

Understanding the class distribution is critical before choosing a classification strategy:
- **Severe imbalance** (>5:1) would require oversampling (SMOTE) or class-weight adjustment
- **Moderate imbalance** is handled by the `class_weight='balanced'` parameter in the DecisionTree base learner
- **Near-balance** is ideal for stacking — the meta-learner does not need reweighting

We check both raw counts and percentage split.

In [ ]:
label_counts = df[TARGET_CLASS].value_counts()
label_pct    = df[TARGET_CLASS].value_counts(normalize=True) * 100

print('=== Class Balance ===')
for lbl in CLASS_LABELS:
    n = label_counts.get(lbl, 0)
    p = label_pct.get(lbl, 0.0)
    print(f'  {lbl:8s}: {n:4d} ({p:.1f}%)')

imbalance_ratio = label_counts.max() / label_counts.min()
print(f'\nMax/Min ratio: {imbalance_ratio:.2f}x')
if imbalance_ratio < 3:
    print('Assessment: ACCEPTABLE — no oversampling needed')
elif imbalance_ratio < 5:
    print('Assessment: MODERATE — consider class_weight="balanced" on base learners')
else:
    print('Assessment: SEVERE — recommend SMOTE or significant class weighting')

colors = {'Human': '#E53935', 'Hybrid': '#FB8C00', 'AI': '#43A047'}
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(CLASS_LABELS,
              [label_counts.get(l, 0) for l in CLASS_LABELS],
              color=[colors[l] for l in CLASS_LABELS],
              width=0.5, edgecolor='white', linewidth=1.5)
for bar, lbl in zip(bars, CLASS_LABELS):
    n = label_counts.get(lbl, 0)
    p = label_pct.get(lbl, 0.0)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
            f'{n}\n({p:.1f}%)', ha='center', fontsize=10)
ax.set_ylabel('Records', fontsize=11)
ax.set_title('Class Balance — target_team_label', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Profit Margin Distribution by Label

The regression target `profit_margin_pct` should vary systematically by class:
- **Human projects** have higher labour/seniority costs → lower margins
- **AI projects** are small/low-risk with tight budgets → higher margins via efficiency
- **Hybrid projects** fall in between

This analysis confirms the regression target carries meaningful signal correlated with (but not identical to) the classification target.

In [ ]:
print('=== Profit Margin by Label ===')
margin_by_label = df.groupby(TARGET_CLASS)[TARGET_REGR].agg(['mean', 'std', 'min', 'max'])
print(margin_by_label.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram overlaid by class
for label, color in [('Human', '#E53935'), ('Hybrid', '#FB8C00'), ('AI', '#43A047')]:
    subset = df[df[TARGET_CLASS] == label][TARGET_REGR]
    axes[0].hist(subset, bins=25, alpha=0.6, label=f'{label} (μ={subset.mean():.1f}%)',
                 color=color, edgecolor='white')
axes[0].set_xlabel('Profit Margin (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('Profit Margin Histogram by Label')
axes[0].legend()

# Box plot
order = CLASS_LABELS
palette = {l: c for l, c in zip(CLASS_LABELS, ['#E53935', '#FB8C00', '#43A047'])}
sns.boxplot(data=df, x=TARGET_CLASS, y=TARGET_REGR, order=order,
            palette=palette, ax=axes[1])
axes[1].set_xlabel('Team Label')
axes[1].set_ylabel('Profit Margin (%)')
axes[1].set_title('Profit Margin Box Plot by Label')

plt.tight_layout()
plt.show()

## 8. Numeric Feature Distributions

We plot histograms for all 5 numeric features. Key observations to look for:

- **`estimated_loc`** follows a log-normal distribution — we expect a right-skewed histogram with a long tail. This suggests a log-transform or robust scaler might be beneficial, though `StandardScaler` is still effective after the long tail is clipped at 200K.
- **`timeline_days`** should be roughly normal, clipped at [7, 365].
- **`testing_coverage_pct`** should show a bell shape centred around 55–75%, increasing with complexity.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i, col in enumerate(NUMERIC_FEATURES):
    axes[i].hist(df[col], bins=35, color='#0288D1', edgecolor='white', alpha=0.85)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.2,
                    label=f'μ={df[col].mean():.1f}')
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('Value', fontsize=8)
    axes[i].legend(fontsize=7)

plt.suptitle('Numeric Feature Distributions', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Skewness check
print('Skewness (|skew| > 1.0 = significant):')
for col in NUMERIC_FEATURES:
    skew = df[col].skew()
    flag = ' ← significantly skewed' if abs(skew) > 1.0 else ''
    print(f'  {col:28s}: {skew:+.2f}{flag}')

## 9. Correlation Heatmap — Numeric Features

We compute Pearson correlations between all numeric features and the regression target. Key findings to look for:

- Strong correlations between `estimated_loc`, `timeline_days`, and `team_size_required` (larger projects need more time and people) — this multicollinearity is handled by `StandardScaler` normalisation and XGBoost's regularisation.
- `profit_margin_pct` should correlate negatively with `estimated_loc` and `team_size_required` (higher cost features reduce margin).
- `testing_coverage_pct` correlates positively with complexity-related features.

In [ ]:
numeric_cols = NUMERIC_FEATURES + [TARGET_REGR]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    mask=mask,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Pearson Correlation — Numeric Features + Profit Margin', fontsize=12)
plt.tight_layout()
plt.show()

# Print top correlations with profit margin
print('\nTop correlations with profit_margin_pct:')
margin_corr = corr[TARGET_REGR].drop(TARGET_REGR).sort_values(key=abs, ascending=False)
for feat, val in margin_corr.items():
    print(f'  {feat:28s}: {val:+.3f}')

## 10. Categorical Features vs. Label — Box Plots and Count Plots

For each key ordinal feature, we visualise the distribution of `profit_margin_pct` at each level, stratified by the team label. This reveals:
- Whether higher-risk/complexity levels consistently produce lower margins (confirming the formula)
- Whether the feature discriminates between Human/Hybrid/AI labels (confirming classifier signal)

In [ ]:
key_features = ['complexity_score', 'technical_risk_level', 'security_criticality', 'budget_pressure']
palette = {'Human': '#E53935', 'Hybrid': '#FB8C00', 'AI': '#43A047'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(key_features):
    levels = ORDINAL_FEATURES[col]
    sns.boxplot(
        data=df,
        x=col, y=TARGET_REGR,
        hue=TARGET_CLASS,
        order=levels,
        hue_order=CLASS_LABELS,
        palette=palette,
        ax=axes[i]
    )
    axes[i].set_title(f'{col} vs profit_margin_pct (by label)', fontsize=10)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Profit Margin (%)')
    axes[i].legend(title='Label', fontsize=8)
    axes[i].tick_params(axis='x', labelrotation=15)

plt.suptitle('Ordinal Features vs. Profit Margin — Stratified by Team Label', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 11. Label Distribution Across Ordinal Levels (Count Plots)

These stacked bar charts reveal how each ordinal level maps to the team label. A feature with **strong label separation** (e.g., High complexity → mostly Human) is a powerful classifier feature. Flat distributions (equal label share at all levels) indicate weak signal.

From the business rules, we expect strong signal in `complexity_score`, `technical_risk_level`, and `security_criticality`.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

palette = {'Human': '#E53935', 'Hybrid': '#FB8C00', 'AI': '#43A047'}

for i, (col, levels) in enumerate(ORDINAL_FEATURES.items()):
    ct = pd.crosstab(df[col], df[TARGET_CLASS]).reindex(levels, fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct[CLASS_LABELS].plot(
        kind='bar',
        stacked=True,
        color=[palette[l] for l in CLASS_LABELS],
        ax=axes[i],
        edgecolor='white',
        linewidth=0.5
    )
    axes[i].set_title(col, fontsize=9)
    axes[i].set_ylabel('% of records')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', labelrotation=25, labelsize=8)
    axes[i].legend(fontsize=7, loc='lower right')
    axes[i].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.suptitle('Label Distribution Across Ordinal Levels (stacked %)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 12. Regulatory Compliance vs. Label

The binary `regulatory_compliance` flag has a hard-coded effect in the label assignment rules: regulated projects with High/Critical risk always get the Human label. We verify this here — regulated projects should have a much higher proportion of Human labels.

In [ ]:
reg_label = df.groupby(['regulatory_compliance', TARGET_CLASS]).size().unstack(fill_value=0)
reg_label_pct = reg_label.div(reg_label.sum(axis=1), axis=0) * 100

print('Label distribution by regulatory_compliance:')
print(reg_label_pct.round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

reg_label_pct[CLASS_LABELS].plot(
    kind='bar',
    stacked=True,
    color=[palette[l] for l in CLASS_LABELS],
    ax=axes[0],
    edgecolor='white'
)
axes[0].set_xticklabels(['Not Regulated (0)', 'Regulated (1)'], rotation=0)
axes[0].set_ylabel('% of records')
axes[0].set_title('Label Mix by Regulatory Compliance')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

# Profit margin by regulatory flag
for flag, color in [(0, '#43A047'), (1, '#E53935')]:
    subset = df[df['regulatory_compliance'] == flag][TARGET_REGR]
    axes[1].hist(subset, bins=25, alpha=0.6,
                 label=f'Regulated={flag} (μ={subset.mean():.1f}%)',
                 color=color, edgecolor='white')
axes[1].set_xlabel('Profit Margin (%)')
axes[1].set_ylabel('Count')
axes[1].set_title('Profit Margin by Regulatory Flag')
axes[1].legend()

plt.tight_layout()
plt.show()

## 13. Domain Category Analysis

The nominal `domain_category` feature captures industry sector context. We expect:
- **Security** domain: disproportionately Human (high security criticality triggers the Human rule)
- **Web / Mobile** domains: more AI-eligible (typically lower complexity/risk)
- **Data** domain: mix of Human and Hybrid due to regulatory exposure

OneHotEncoder with `drop='first'` is used in preprocessing, so Web (the first alphabetically when sorted, or as configured) becomes the implicit baseline.

In [ ]:
domain_label = df.groupby(['domain_category', TARGET_CLASS]).size().unstack(fill_value=0)
domain_label_pct = domain_label.reindex(DOMAIN_CATEGORIES).div(
    domain_label.reindex(DOMAIN_CATEGORIES).sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

domain_label_pct[CLASS_LABELS].plot(
    kind='bar', stacked=True,
    color=[palette[l] for l in CLASS_LABELS],
    ax=axes[0], edgecolor='white'
)
axes[0].set_xticklabels(DOMAIN_CATEGORIES, rotation=15)
axes[0].set_ylabel('% of records')
axes[0].set_title('Label Distribution by Domain Category')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].legend(title='Label')

# Margin by domain
domain_margin = df.groupby('domain_category')[TARGET_REGR].mean().reindex(DOMAIN_CATEGORIES)
axes[1].bar(DOMAIN_CATEGORIES, domain_margin.values,
            color='#7E57C2', edgecolor='white', alpha=0.85)
for j, v in enumerate(domain_margin.values):
    axes[1].text(j, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)
axes[1].set_ylabel('Mean Profit Margin (%)')
axes[1].set_title('Mean Profit Margin by Domain')

plt.tight_layout()
plt.show()

## 14. Feature Distribution Analysis — Numeric vs. Label

We use violin plots to compare the distribution of each numeric feature across the three label classes. Violin plots are preferred over box plots here because they reveal multi-modal distributions (e.g., `estimated_loc` being bimodal within the Hybrid class, reflecting both small and large projects that fall in the middle ground).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

palette_list = {'Human': '#E53935', 'Hybrid': '#FB8C00', 'AI': '#43A047'}

for i, col in enumerate(NUMERIC_FEATURES):
    sns.violinplot(
        data=df, x=TARGET_CLASS, y=col,
        order=CLASS_LABELS,
        palette=palette_list,
        inner='box',
        ax=axes[i]
    )
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', labelrotation=10, labelsize=8)

plt.suptitle('Numeric Feature Distributions by Team Label', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 15. Pairplot — Key Feature Interactions

A pairplot of the three most discriminative features (based on business-rule analysis) coloured by label class. This reveals whether linear decision boundaries are sufficient or whether non-linear models (XGBoost, ANN) are required.

We sub-sample to 200 records for rendering speed — representative of the full distribution given random stratified sampling.

In [ ]:
# Encode ordinal columns numerically for pairplot
ordinal_map = {
    'complexity_score': {'Low': 0, 'Medium': 1, 'High': 2},
    'technical_risk_level': {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3},
    'security_criticality': {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3},
}

df_pp = df[['estimated_loc', 'complexity_score', 'technical_risk_level',
            'security_criticality', TARGET_CLASS]].copy()
for col, mapping in ordinal_map.items():
    df_pp[col] = df_pp[col].map(mapping)

# Sub-sample 200 records
df_sample = df_pp.groupby(TARGET_CLASS, group_keys=False).apply(
    lambda g: g.sample(min(len(g), 66), random_state=42)
).reset_index(drop=True)

g = sns.pairplot(
    df_sample,
    hue=TARGET_CLASS,
    hue_order=CLASS_LABELS,
    palette={'Human': '#E53935', 'Hybrid': '#FB8C00', 'AI': '#43A047'},
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 's': 25}
)
g.figure.suptitle('Pairplot — LOC, Complexity, Risk, Security (coloured by label)', y=1.02, fontsize=11)
plt.show()

## 16. Key EDA Findings

| Finding | Implication for Modelling |
|---|---|
| `estimated_loc` is strongly right-skewed (log-normal) | StandardScaler normalises this after the skew; XGBoost is rank-invariant so unaffected |
| `complexity_score`, `technical_risk_level`, `security_criticality` have the strongest label separation | XGBoost feature importance should rank these highest (confirmed in Notebook 04) |
| Class imbalance is moderate (Human is smallest class) | Handled by `class_weight='balanced'` in DT and `stratify=y` in train/test split |
| `regulatory_compliance` = 1 → almost exclusively Human | Binary feature has strong signal despite being 0/1; PassThrough is the right treatment |
| Security domain has highest Human proportion | Domain OHE adds signal beyond the ordinal security level |
| Profit margin is clearly separable by label class | Regression model can use same features as classifier without leakage |
| Low pairwise correlation among numeric features (< 0.6) | No need for PCA or feature removal; all 5 numeric features contribute independently |

**Next step:** Notebook 03 — Feature Engineering (`03_feature_engineering.ipynb`)